In [1]:
# import os
# import wandb
# from kaggle_secrets import UserSecretsClient

# # fetch secret
# user_secrets = UserSecretsClient()
# wandb_key = user_secrets.get_secret("WANDB_API_KEY")

# # set environment variable
# os.environ["WANDB_API_KEY"] = wandb_key

# # login
# wandb.login(relogin=True)

In [2]:
from pathlib import Path

MESSY_DATA_DIR = Path("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup")
ESC_50_MASTER = MESSY_DATA_DIR / "ESC-50-master"
GENRES_STEMS = MESSY_DATA_DIR / "genres_stems"
MASHUPS = MESSY_DATA_DIR / "mashups"

In [3]:
import os 
import requests
from pathlib import Path

Path("pretrained_models").mkdir(exist_ok=True)

url = "https://zenodo.org/records/3987831/files/Cnn14_mAP%3D0.431.pth"

output_path = "pretrained_models/Cnn14.pth"

with requests.get(url, stream=True) as r:
    r.raise_for_status()
    with open(output_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

print("Download complete")

Download complete


In [4]:
import os
os.environ["TORCHAUDIO_USE_BACKEND_DISPATCHER"] = "0"

from pathlib import Path
from torch.utils.data import Dataset
import torchaudio

import pandas as pd
import torch


# Took entire day of making the dataloder and debuging it :|

'''
Nyquist theorem:

Max frequency representable = sample_rate / 2
I learned it in sns class

our sample rate is  44.1 kHz, mono

so sample max frequency representable = 22050


currently we have time domain signal 
converts it in frequency domain and it can be done using Fourier Transform (STFT)
'''

# ORIGINAL DATALOADERS (with preprocessing - for training from scratch)

class Variables:
    SAMPLE_RATE = 22050
    N_MELS = 256
    DURATION = 5   # seconds
    SAMPLES = SAMPLE_RATE * DURATION

    def __init__(self):
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=self.SAMPLE_RATE,
            n_fft=1024,
            hop_length=256,
            n_mels=self.N_MELS,
            f_min=20,
            f_max=8000,
            power=2.0
        )

        # dB=10⋅log10​(Power)
        self.db = torchaudio.transforms.AmplitudeToDB(top_db=80)

    def _fix_length(self, signal: torch.Tensor):
        length = signal.shape[1]

        # too long → crop
        if length > self.SAMPLES:
            signal = signal[:, :self.SAMPLES]

        # too short → pad
        elif length < self.SAMPLES:
            padding = self.SAMPLES - length
            signal = torch.nn.functional.pad(signal, (0, padding))

        return signal


class ESC50_DataLoader(Dataset):
    def __init__(self, esc50_dir: Path, allowed_folds=None):
        self.esc50_dir = esc50_dir
        self.audio_dir = esc50_dir / "audio"
        self.variables = Variables()

        self.meta_data = None
        self.allowed_folds = allowed_folds

    def load_meta_data(self, meta_data_path: str):
        print(f"Loading metadata from file: {meta_data_path}")
        df = pd.read_csv(meta_data_path)

        # filter folds
        if self.allowed_folds is not None:
            df = df[df["fold"].isin(self.allowed_folds)].reset_index(drop=True)

        self.meta_data = df
    
    
    def __len__(self):
        return self.meta_data.shape[0]

    
    def __getitem__(self, index: int):
        row = self.meta_data.iloc[index]

        filename = row["filename"]
        label = int(row["target"])

        file_path = self.audio_dir / filename

        waveform, sr = torchaudio.load(file_path)

        # mono as esc50 is has mono channel
        waveform = torch.mean(waveform, dim=0, keepdim=True)

        # resample
        if sr != self.variables.SAMPLE_RATE:
            resampler = torchaudio.transforms.Resample(sr, self.variables.SAMPLE_RATE)
            waveform = resampler(waveform)

        waveform = self.variables._fix_length(signal=waveform)

        # mel spectrogram
        mel = self.variables.mel(waveform)
        mel = self.variables.db(mel)

        # Convert [-80, 0] dB → [0, 1]
        mel = (mel + 80) / 80

        # clamp for safety
        mel = torch.clamp(mel, 0, 1)

        return mel, torch.tensor(label, dtype=torch.long)


class GenreDataLoader(Dataset):
    def __init__(self, root_dir: Path):
        self.root_dir = root_dir 
        self.variables = Variables()

        self.samples = []
        self.labels = []
        self.class_map = {}

        self._load_files()


    def _load_files(self):

        for label_id, genre_dir in enumerate(sorted(self.root_dir.iterdir())):

            if not genre_dir.is_dir():
                continue

            genre_name = genre_dir.name 
            self.class_map[genre_name] = label_id

            # each subfolder = One song
            for song_dir in genre_dir.iterdir():

                if not song_dir.is_dir():
                    continue
                
                # collect all wav parts of that song
                wav_parts = sorted(list(song_dir.glob("*.wav")))
                # print(wav_parts)
                if len(wav_parts) == 0:
                    continue
                
                self.samples.append(wav_parts)
                self.labels.append(label_id)

        print("Classes:", self.class_map)
        print("Total songs:", len(self.samples))


    def __len__(self):
        return len(self.samples)
    

    def __getitem__(self, index):
        '''
        Docstring for __getitem__
        
        I am combining each .wav file for songs to get a final waveform for training for 5 sec window
        '''

        wave_list = self.samples[index]
        label = self.labels[index]

        waveforms = []

        for wav_path in wave_list:
            waveform, sr = torchaudio.load(wav_path)

            # mono
            waveform = torch.mean(waveform, dim=0, keepdim=True)

            # resample
            if sr != self.variables.SAMPLE_RATE:
                resampler = torchaudio.transforms.Resample(sr, self.variables.SAMPLE_RATE)
                waveform = resampler(waveform)

            waveform = self.variables._fix_length(signal=waveform)

            waveforms.append(waveform)

        final_waveform = torch.stack(waveforms, dim=0).mean(dim=0)

        # final_waveform = self.variables._fix_length(signal=final_waveform)
        # print(final_waveform.shape)
        
        mel = self.variables.mel(final_waveform)
        mel = self.variables.db(mel)

        mel = (mel + 80) / 80
        mel = torch.clamp(mel, 0, 1)

        return mel, torch.tensor(label, dtype=torch.long)



# DATALOADERS (no preprocessing - for finetuning pretrained CNN14)

class ESC50_DataLoader_Waveform(Dataset):
    def __init__(self, esc50_dir: Path, allowed_folds=None):
        self.esc50_dir = esc50_dir
        self.audio_dir = esc50_dir / "audio"
        
        self.SAMPLE_RATE = 32000
        self.DURATION = 5
        
        self.SAMPLES = self.SAMPLE_RATE * self.DURATION
        
        self.meta_data = None
        self.allowed_folds = allowed_folds

    def load_meta_data(self, meta_data_path: str):
        print(f"Loading metadata from file: {meta_data_path}")
        df = pd.read_csv(meta_data_path)

        if self.allowed_folds is not None:
            df = df[df["fold"].isin(self.allowed_folds)].reset_index(drop=True)

        self.meta_data = df
    
    def __len__(self):
        return self.meta_data.shape[0]

    def _fix_length(self, signal: torch.Tensor):
        """Fix signal length to match CNN14 requirements (10 seconds at 32kHz)"""
        length = signal.shape[1]

        if length > self.SAMPLES:
            signal = signal[:, :self.SAMPLES]

        elif length < self.SAMPLES:
            padding = self.SAMPLES - length
            signal = torch.nn.functional.pad(signal, (0, padding))

        return signal
    
    def __getitem__(self, index: int):
        row = self.meta_data.iloc[index]
        filename = row["filename"]
        label = int(row["target"])
        file_path = self.audio_dir / filename

        waveform, sr = torchaudio.load(file_path)

        waveform = torch.mean(waveform, dim=0, keepdim=True)

        if sr != self.SAMPLE_RATE:
            resampler = torchaudio.transforms.Resample(sr, self.SAMPLE_RATE)
            waveform = resampler(waveform)

        waveform = self._fix_length(signal=waveform)

        return waveform, torch.tensor(label, dtype=torch.long)


class GenreDataLoader_Waveform_Chunked(Dataset):
    def __init__(self, root_dir: Path):
        self.root_dir = root_dir 
        
        # CNN14 requirements
        self.SAMPLE_RATE = 32000
        self.DURATION = 5
        self.CHUNK_SAMPLES = self.SAMPLE_RATE * self.DURATION

        self.samples = []
        self.labels = []
        self.class_map = {}

        self._load_files()

    def _load_files(self):
        for label_id, genre_dir in enumerate(sorted(self.root_dir.iterdir())):

            if not genre_dir.is_dir():
                continue

            genre_name = genre_dir.name 
            self.class_map[genre_name] = label_id

            # each subfolder = One song
            for song_dir in genre_dir.iterdir():

                if not song_dir.is_dir():
                    continue
                
                # collect all wav parts of that song
                wav_parts = sorted(list(song_dir.glob("*.wav")))
                if len(wav_parts) == 0:
                    continue
                
                self.samples.append(wav_parts)
                self.labels.append(label_id)

        print("Classes:", self.class_map)
        print("Total songs:", len(self.samples))

    def __len__(self):
        return len(self.samples)
    
    def _fix_length(self, signal: torch.Tensor) -> torch.Tensor:
        length = signal.shape[1]

        # too long → crop
        if length > self.CHUNK_SAMPLES:
            signal = signal[:, :self.CHUNK_SAMPLES]

        # too short → pad with zeros
        elif length < self.CHUNK_SAMPLES:
            padding = self.CHUNK_SAMPLES - length
            signal = torch.nn.functional.pad(signal, (0, padding))

        return signal
    
    def __getitem__(self, index):
        wave_list = self.samples[index]
        label = self.labels[index]

        waveforms = []

        for wav_path in wave_list:
            waveform, sr = torchaudio.load(wav_path)

            waveform = torch.mean(waveform, dim=0, keepdim=True)

            if sr != self.SAMPLE_RATE:
                resampler = torchaudio.transforms.Resample(sr, self.SAMPLE_RATE)
                waveform = resampler(waveform)

            waveform = self._fix_length(signal=waveform)

            waveforms.append(waveform)

        full_waveform = torch.cat(waveforms, dim=1)

        total_samples = full_waveform.shape[1]
        num_chunks = total_samples // self.CHUNK_SAMPLES

        if num_chunks == 0:
            # Song is shorter than 5 seconds, pad it
            full_waveform = self._fix_length(full_waveform, self.CHUNK_SAMPLES)
            chunks = full_waveform.unsqueeze(0)  # [1, 1, 160000]

        else:
            # Extract complete chunks (30 sec → 6 chunks of 5 sec)
            chunks = []
            for i in range(num_chunks):
                start_idx = i * self.CHUNK_SAMPLES
                end_idx = start_idx + self.CHUNK_SAMPLES

                chunk = full_waveform[:, start_idx: end_idx]
                chunks.append(chunk)
            
            chunks = torch.stack(chunks, dim=0)

        return chunks, torch.tensor(label, dtype=torch.long)
    


def collate_fn_chunks(batch):
    """
    Custom collate function for chunked genre data
    Handles variable number of chunks per song
    """
    all_chunks = []
    labels = []
    chunk_counts = []
    
    for chunks, label in batch:
        num_chunks = chunks.shape[0]
        all_chunks.append(chunks)
        labels.extend([label] * num_chunks)
        chunk_counts.append(num_chunks)
    
    all_chunks = torch.cat(all_chunks, dim=0)  # [total_chunks, 1, 160000]
    labels = torch.tensor(labels, dtype=torch.long)
    
    return all_chunks, labels, chunk_counts


In [5]:
import torch 
import torch.nn as nn
from torch.utils.data import DataLoader, random_split, WeightedRandomSampler
from collections import Counter

import numpy as np

# from helper.custom_dataloader import ESC50_DataLoader_Waveform, GenreDataLoader_Waveform_Chunked, collate_fn_chunks
# import wandb


def train_esc50(model, dataloader, loss_function, optimizer, device):

    model.to(device)

    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for waveform, label in dataloader:  
        waveform = waveform.to(device)
        label = label.to(device)

        # Forward
        output_dict = model(waveform)
        logits = output_dict["clipwise_output"]

        # Loss
        loss = loss_function(logits, label)

        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Stats
        running_loss += loss.item()
        _, predicted = torch.max(logits, 1)
        correct += (predicted == label).sum().item()
        total += label.size(0)

    epoch_loss = running_loss / len(dataloader)
    accuracy = 100 * correct / total

    return epoch_loss, accuracy


def test_esc50(model, dataloader, loss_function, device):

    model.to(device)
    
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for spectrum, label in dataloader:

            spectrum = spectrum.to(device)
            label = label.to(device)

            # forward
            # Remove the None parameter - model expects only input
            output_dict = model(spectrum)
            logits = output_dict["clipwise_output"]

            loss = loss_function(logits, label)

            running_loss += loss.item()

            _, predicted = torch.max(logits, 1)
            correct += (predicted == label).sum().item()
            total += label.size(0)

    loss = running_loss / len(dataloader) if len(dataloader) > 0 else 0
    accuracy = 100 * correct / total if total > 0 else 0

    return loss, accuracy



def train_esc50_model(model, device, train_dataloader, test_dataloader, epochs=100):
    print(f"\n{'='*70}")
    print(f"ESC-50 TRAINING")
    print(f"{'='*70}\n")
    
    # Clear GPU memory
    if device == "cuda":
        torch.cuda.empty_cache()


    # wandb.init(
    #     project=f"esc50-classification",
    #     name=f"Traning finetune",
    #     config={
    #         "epochs": epochs,
    #         "batch_size": batch_size,
    #         "optimizer": "Adam",
    #         "dataset": "ESC-50",
    #         "architecture": model.__class__.__name__,
    #     }
    # )

    # wandb.watch(model, log="all", log_freq=50)

    print("\n3. Setting up training...")
    model = model.to(device)
    loss_function = nn.CrossEntropyLoss()
    
    # Initial optimizer
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=0.001
    )

    change_stage = epochs // 3

    best_val_acc = 0
    for epoch in range(epochs):
        if epoch == 0:
            # STAGE 1: Only train final layer
            for name, param in model.named_parameters():
                if 'fc_audioset' not in name:
                    param.requires_grad = False
            
            optimizer = torch.optim.Adam(
                filter(lambda p: p.requires_grad, model.parameters()),
                lr=0.001
            )
            trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

            print(f"STAGE 1: Training only final layer (lr=0.001)")
            print(f"         Trainable params: {trainable:,}")
            print(f"{'='*70}\n")
        
        elif epoch == change_stage:
            # STAGE 2: Unfreeze last 2 conv blocks
            for name, param in model.named_parameters():
                if 'conv_block5' in name or 'conv_block6' in name or 'fc' in name:
                    param.requires_grad = True
            
            optimizer = torch.optim.Adam(
                filter(lambda p: p.requires_grad, model.parameters()),
                lr=0.0001
            )
            trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

            print(f"\n{'='*70}")
            print(f"STAGE 2: Unfroze conv_block5, conv_block6 (lr=0.0001)")
            print(f"         Trainable params: {trainable:,}")
            print(f"{'='*70}\n")
        
        elif epoch == change_stage * 2:
            # STAGE 3: Unfreeze all conv blocks
            for name, param in model.named_parameters():
                if 'spectrogram' not in name and 'logmel' not in name:
                    param.requires_grad = True
            
            optimizer = torch.optim.Adam(
                filter(lambda p: p.requires_grad, model.parameters()),
                lr=0.00001
            )
            trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

            print(f"\n{'='*70}")
            print(f"STAGE 3: Unfroze all conv blocks (lr=0.00001)")
            print(f"         Trainable params: {trainable:,}")
            print(f"{'='*70}\n")
        
        train_loss, train_acc = train_esc50(
            model=model,
            dataloader=train_dataloader,
            loss_function=loss_function,
            optimizer=optimizer,
            device=device
        )
        
        val_loss, val_acc = test_esc50(
            model=model,
            dataloader=test_dataloader,
            loss_function=loss_function,
            device=device
        )
        
        print(
            f"Epoch [{epoch+1:3d}/{epochs}] "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:6.2f}% | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:6.2f}%"
        )

        # wandb.log({
        #     "epoch": epoch + 1,
        #     "train_loss": train_loss,
        #     "train_accuracy": train_acc,
        #     "val_loss": val_loss,
        #     "val_accuracy": val_acc,
        #     "lr": optimizer.param_groups[0]["lr"]
        # })

        if val_acc >= best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), "best_esc50_model.pth")
            print(f"         ✓ Saved best model (val_acc: {val_acc:.2f}%)")
            # wandb.save("best_esc50_model.pth")

    print(f"\n{'='*70}")
    print(f"ESC-50 TRAINING COMPLETE")
    print(f"{'='*70}")
    print(f"Best Validation Accuracy: {best_val_acc:.2f}%")
    print(f"Model saved as: best_esc50_model.pth")
    print(f"{'='*70}\n")
    
    return best_val_acc





def train_genre(model, dataloader, loss_function, optimizer, device):

    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for chunks, labels, chunk_counts in dataloader:
        # chunks: [total_chunks_in_batch, 1, 160000]
        # labels: [total_chunks_in_batch] - repeated for each chunk
        # chunk_counts: [batch_size] - number of chunks per song
        
        chunks = chunks.to(device)
        labels = labels.to(device)

        output_dict = model(chunks)
        logits = output_dict["clipwise_output"]  # [total_chunks, num_genres]

        song_logits = []
        start_idx = 0
        true_labels = []

        for count in chunk_counts:
            end_idx = start_idx + count
            
            # Get this song's chunk predictions
            song_chunk_logits = logits[start_idx:end_idx]  # [6, num_genres]
            
            # Average across the 6 chunks
            avg_logits = song_chunk_logits.mean(dim=0, keepdim=True)  # [1, num_genres]
            song_logits.append(avg_logits)
            
            # Get true label (same for all chunks)
            true_labels.append(labels[start_idx])
            
            start_idx = end_idx

        song_logits = torch.cat(song_logits, dim=0)  # [batch_size, num_genres]
        true_labels = torch.stack(true_labels)        # [batch_size]
        
        # Loss on averaged predictions
        loss = loss_function(song_logits, true_labels)

        # Backward (same as ESC-50)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(song_logits, 1)
        correct += (predicted == true_labels).sum().item()
        total += true_labels.size(0)

    epoch_loss = running_loss / len(dataloader)
    accuracy = 100 * correct / total

    return epoch_loss, accuracy


def test_genre(model, dataloader, loss_function, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for chunks, labels, chunk_counts in dataloader:
            chunks = chunks.to(device)
            labels = labels.to(device)

            output_dict = model(chunks)
            logits = output_dict["clipwise_output"]
            
            # Average predictions per song
            song_logits = []
            start_idx = 0
            true_labels = []
            
            for count in chunk_counts:
                end_idx = start_idx + count

                song_chunk_logits = logits[start_idx:end_idx]
                avg_logits = song_chunk_logits.mean(dim=0, keepdim=True)

                song_logits.append(avg_logits)
                true_labels.append(labels[start_idx])
                start_idx = end_idx
            
            song_logits = torch.cat(song_logits, dim=0)
            true_labels = torch.stack(true_labels)
            
            loss = loss_function(song_logits, true_labels)
            running_loss += loss.item()

            _, predicted = torch.max(song_logits, 1)
            correct += (predicted == true_labels).sum().item()
            total += true_labels.size(0)

    loss = running_loss / len(dataloader) if len(dataloader) > 0 else 0
    accuracy = 100 * correct / total if total > 0 else 0

    return loss, accuracy


def train_genre_model(model, device, train_dataloader, test_dataloader, epochs=50):
    print(f"\n{'='*70}")
    print(f"GENRE CLASSIFICATION TRAINING")
    print(f"{'='*70}\n")
    
    # Clear GPU memory
    if device == "cuda":
        torch.cuda.empty_cache()


    # wandb.init(
    #     project=f"genre-classification",
    #     name=f"Traning finetune",
    #     config={
    #         "epochs": epochs,
    #         "batch_size": batch_size,
    #         "optimizer": "Adam",
    #         "dataset": "ESC-50",
    #         "architecture": model.__class__.__name__,
    #     }
    # )

    # wandb.watch(model, log="all", log_freq=50)

    print("\n3. Setting up training...")
    model = model.to(device)

    loss_function = nn.CrossEntropyLoss()
    
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=0.001
    )

    best_val_acc = 0
    
    print(f"\n4. Starting training for {epochs} epochs...")
    print(f"{'='*70}\n")

    change_stage = epochs // 3

    for epoch in range(epochs):
        if epoch == 0:
            # STAGE 1: Only train final layer
            for name, param in model.named_parameters():
                if 'fc_audioset' not in name:
                    param.requires_grad = False
            
            optimizer = torch.optim.Adam(
                filter(lambda p: p.requires_grad, model.parameters()),
                lr=0.001
            )
            trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

            print(f"STAGE 1: Training only final layer (lr=0.001)")
            print(f"         Trainable params: {trainable:,}")
            print(f"{'='*70}\n")
        
        elif epoch == change_stage:
            # STAGE 2: Unfreeze last 2 conv blocks
            for name, param in model.named_parameters():
                if 'conv_block5' in name or 'conv_block6' in name or 'fc' in name:
                    param.requires_grad = True
            
            optimizer = torch.optim.Adam(
                filter(lambda p: p.requires_grad, model.parameters()),
                lr=0.0001
            )

            trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
            print(f"\n{'='*70}")
            print(f"STAGE 2: Unfroze conv_block5, conv_block6 (lr=0.0001)")
            print(f"         Trainable params: {trainable:,}")
            print(f"{'='*70}\n")
        
        elif epoch == change_stage * 2:
            # STAGE 3: Unfreeze all conv blocks
            for name, param in model.named_parameters():
                if 'spectrogram' not in name and 'logmel' not in name:
                    param.requires_grad = True
            
            optimizer = torch.optim.Adam(
                filter(lambda p: p.requires_grad, model.parameters()),
                lr=0.00005
            )

            trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
            print(f"\n{'='*70}")
            print(f"STAGE 3: Unfroze all conv blocks (lr=0.00005)")
            print(f"         Trainable params: {trainable:,}")
            print(f"{'='*70}\n")

        train_loss, train_acc = train_genre(
            model=model,
            dataloader=train_dataloader,
            loss_function=loss_function,
            optimizer=optimizer,
            device=device
        )
        
        val_loss, val_acc = test_genre(
            model=model,
            dataloader=test_dataloader,
            loss_function=loss_function,
            device=device
        )

        print(
            f"Epoch [{epoch+1:3d}/{epochs}] "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:6.2f}% | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:6.2f}%"
        )

        # wandb.log({
        #     "epoch": epoch + 1,
        #     "train_loss": train_loss,
        #     "train_accuracy": train_acc,
        #     "val_loss": val_loss,
        #     "val_accuracy": val_acc,
        #     "lr": optimizer.param_groups[0]["lr"]
        # })
        
        if val_acc >= best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), "best_genre_model.pth")
            print(f"         ✓ Saved best model (val_acc: {val_acc:.2f}%)")
            # wandb.save("best_genre_model.pth")

    print(f"\n{'='*70}")
    print(f"GENRE CLASSIFICATION TRAINING COMPLETE")
    print(f"{'='*70}")
    print(f"Best Validation Accuracy: {best_val_acc:.2f}%")
    print(f"Model saved as: best_genre_model.pth")
    print(f"{'='*70}\n")
    
    return best_val_acc


In [6]:
! pip install torchlibrosa

In [7]:
import torch 
import torch.nn as nn 
import torch.nn.functional as F
from torchlibrosa.stft import Spectrogram, LogmelFilterBank 
from torchlibrosa.augmentation import SpecAugmentation 


"""
I copied this model arch form externaly
"""


def init_layer(layer):
    """Initialize a Linear or Convolutional layer."""
    nn.init.xavier_uniform_(layer.weight)
    if hasattr(layer, 'bias'):
        if layer.bias is not None:
            layer.bias.data.fill_(0.)


def init_bn(bn):
    """Initialize a Batchnorm layer."""
    bn.bias.data.fill_(0.)
    bn.weight.data.fill_(1.)


def do_mixup(x, mixup_lambda):
    """Mixup x of even indexes (0, 2, 4, ...) with x of odd indexes (1, 3, 5, ...)."""
    out = (x[0::2].transpose(0, -1) * mixup_lambda[0::2] + \
        x[1::2].transpose(0, -1) * mixup_lambda[1::2]).transpose(0, -1)
    return out



class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ConvBlock, self).__init__()
        
        self.conv1 = nn.Conv2d(in_channels=in_channels, 
                              out_channels=out_channels,
                              kernel_size=(3, 3), stride=(1, 1),
                              padding=(1, 1), bias=False)
                              
        self.conv2 = nn.Conv2d(in_channels=out_channels, 
                              out_channels=out_channels,
                              kernel_size=(3, 3), stride=(1, 1),
                              padding=(1, 1), bias=False)
                              
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.bn2 = nn.BatchNorm2d(out_channels)

        init_layer(self.conv1)
        init_layer(self.conv2)
        init_bn(self.bn1)
        init_bn(self.bn2)

    def forward(self, input, pool_size=(2, 2), pool_type='avg'):
        x = input
        x = F.relu_(self.bn1(self.conv1(x)))
        x = F.relu_(self.bn2(self.conv2(x)))
        if pool_type == 'max':
            x = F.max_pool2d(x, kernel_size=pool_size)
        elif pool_type == 'avg':
            x = F.avg_pool2d(x, kernel_size=pool_size)
        elif pool_type == 'avg+max':
            x1 = F.avg_pool2d(x, kernel_size=pool_size)
            x2 = F.max_pool2d(x, kernel_size=pool_size)
            x = x1 + x2
        else:
            raise Exception('Incorrect argument!')
        
        return x


class Cnn14(nn.Module):
    def __init__(self, classes_num):
        super(Cnn14, self).__init__()

        # Spectrogram parameters for 5-second audio
        window_size = 1024
        hop_size = 320
        mel_bins = 64
        fmin = 50
        fmax = 14000
        sample_rate = 32000
        
        window = 'hann'
        center = True
        pad_mode = 'reflect'
        ref = 1.0
        amin = 1e-10
        top_db = None

        # Feature extraction layers
        self.spectrogram_extractor = Spectrogram(
            n_fft=window_size, 
            hop_length=hop_size,
            win_length=window_size, 
            window=window, 
            center=center, 
            pad_mode=pad_mode,
            freeze_parameters=True
        )

        self.logmel_extractor = LogmelFilterBank(
            sr=sample_rate, 
            n_fft=window_size,
            n_mels=mel_bins, 
            fmin=fmin, 
            fmax=fmax, 
            ref=ref, 
            amin=amin, 
            top_db=top_db,
            freeze_parameters=True
        )

        self.spec_augmenter = SpecAugmentation(
            time_drop_width=64,
            time_stripes_num=2, 
            freq_drop_width=8,
            freq_stripes_num=2
        )

        self.bn0 = nn.BatchNorm2d(64)

        self.conv_block1 = ConvBlock(in_channels=1, out_channels=64)
        self.conv_block2 = ConvBlock(in_channels=64, out_channels=128)
        self.conv_block3 = ConvBlock(in_channels=128, out_channels=256)
        self.conv_block4 = ConvBlock(in_channels=256, out_channels=512)
        self.conv_block5 = ConvBlock(in_channels=512, out_channels=1024)
        self.conv_block6 = ConvBlock(in_channels=1024, out_channels=2048)

        self.fc1 = nn.Linear(2048, 2048, bias=True)
        self.fc_audioset = nn.Linear(2048, classes_num, bias=True)

        init_bn(self.bn0)
        init_layer(self.fc1)
        init_layer(self.fc_audioset)

    def forward(self, input, mixup_lambda=None):
        """
        Input: (batch_size, 1, 160000) or (batch_size, 160000)
        5 seconds at 32kHz = 160,000 samples
        """
        # Handle both 2D and 3D input
        if input.dim() == 3:
            input = input.squeeze(1)  # [batch, 1, 160000] -> [batch, 160000]
        
        # Extract features
        x = self.spectrogram_extractor(input)   # [batch, 1, time_steps, freq_bins]
        x = self.logmel_extractor(x)            # [batch, 1, time_steps, mel_bins]

        x = x.transpose(1, 3)
        x = self.bn0(x)
        x = x.transpose(1, 3)

        if self.training:
            x = self.spec_augmenter(x)

        if self.training and mixup_lambda is not None:
            x = do_mixup(x, mixup_lambda)

        x = self.conv_block1(x, pool_size=(2, 2), pool_type='avg')
        x = F.dropout(x, p=0.2, training=self.training)
        x = self.conv_block2(x, pool_size=(2, 2), pool_type='avg')
        x = F.dropout(x, p=0.2, training=self.training)
        x = self.conv_block3(x, pool_size=(2, 2), pool_type='avg')
        x = F.dropout(x, p=0.2, training=self.training)
        x = self.conv_block4(x, pool_size=(2, 2), pool_type='avg')
        x = F.dropout(x, p=0.2, training=self.training)
        x = self.conv_block5(x, pool_size=(2, 2), pool_type='avg')
        x = F.dropout(x, p=0.2, training=self.training)
        x = self.conv_block6(x, pool_size=(1, 1), pool_type='avg')
        x = F.dropout(x, p=0.2, training=self.training)
        x = torch.mean(x, dim=3)

        (x1, _) = torch.max(x, dim=2)
        x2 = torch.mean(x, dim=2)
        x = x1 + x2
        x = F.dropout(x, p=0.5, training=self.training)
        x = F.relu_(self.fc1(x))
        embedding = F.dropout(x, p=0.5, training=self.training)
        
        # IMPORTANT: No sigmoid! Return raw logits for CrossEntropyLoss
        clipwise_output = self.fc_audioset(x)

        output_dict = {
            'clipwise_output': clipwise_output, 
            'embedding': embedding
        }

        return output_dict
    

In [8]:
from collections import Counter

from pathlib import Path
import torch 
import torch.nn as nn 
from torch.utils.data import WeightedRandomSampler, DataLoader, random_split

# from finetune.utils import train_esc50_model, train_genre_model
# from finetune.cnn14 import Cnn14

# from helper.custom_dataloader import (
#     GenreDataLoader_Waveform_Chunked,
#     ESC50_DataLoader_Waveform,
#     collate_fn_chunks
# )


"""
first train cnn14 on esc50 data 
then train on genure data
"""

class ModelPipeline:
    def __init__(self, esc50_dir: Path, genre_dir: Path, esc50_epochs:int=30, genre_epochs:int=30, esc50_batch_size: int=64, genre_batch_size: int=32):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.esc50_dir = esc50_dir
        self.genre_dir = genre_dir
        self.esc50_epochs = esc50_epochs
        self.genre_epochs = genre_epochs
        self.esc50_batch_size = esc50_batch_size
        self.genre_batch_size = genre_batch_size

        self.esc50_num_classes = 50
        self.num_genres = 10

    def get_esc50_dataloader(self):
        print("Loading ESC-50 dataset...")
        train_dataset = ESC50_DataLoader_Waveform(esc50_dir=self.esc50_dir, allowed_folds=[1, 2, 3, 4])
        train_dataset.load_meta_data(self.esc50_dir / "meta" / "esc50.csv")
        test_dataset = ESC50_DataLoader_Waveform(esc50_dir=self.esc50_dir, allowed_folds=[5])
        test_dataset.load_meta_data(self.esc50_dir / "meta" / "esc50.csv")

        train_loader = DataLoader(
            dataset=train_dataset, 
            batch_size=self.esc50_batch_size, 
            shuffle=True
        )
        test_loader = DataLoader(
            dataset=test_dataset, 
            batch_size=self.esc50_batch_size, 
            shuffle=False
        )

        return train_loader, test_loader
    

    def get_genre_dataloader(self):
        print("Loading genre dataset...")
        genre_dataset = GenreDataLoader_Waveform_Chunked(root_dir=self.genre_dir)

        train_size = int(0.9 * len(genre_dataset))
        val_size = len(genre_dataset) - train_size
        train_dataset, val_dataset = random_split(
            genre_dataset,
            [train_size, val_size],
            generator=torch.Generator().manual_seed(42)
        )

        train_indices = train_dataset.indices
        train_labels = [genre_dataset.labels[i] for i in train_indices]

        class_counts = Counter(train_labels)

        class_weights = {cls: 1.0 / count for cls, count in class_counts.items()}
        sample_weights = [class_weights[label] for label in train_labels]

        sampler = WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(sample_weights),
            replacement=True
        )

        print(f"Train dataset: {len(train_dataset)} samples")
        print(f"Val dataset: {len(val_dataset)} samples")
        print("Class distribution in training:")
        genre_names = {v: k for k, v in genre_dataset.class_map.items()}
        for cls, count in sorted(class_counts.items()):
            print(f"  {genre_names[cls]}: {count} songs (weight: {class_weights[cls]:.3f})")

        train_dataloader = DataLoader(
            dataset=train_dataset,
            batch_size=self.genre_batch_size,
            sampler=sampler,  # NO shuffle when using sampler!
            collate_fn=collate_fn_chunks,
            num_workers=4
        )

        test_dataloader = DataLoader(
            dataset=val_dataset,
            batch_size=self.genre_batch_size,
            shuffle=False,
            collate_fn=collate_fn_chunks,
            num_workers=4
        )

        return train_dataloader, test_dataloader


    def load_pretrained_cnn14(self):

        print("\n" + "="*70)
        print("LOADING PRETRAINED CNN14 FOR ESC-50")
        print("="*70)

        model = Cnn14(classes_num=527)
        
        checkpoint = torch.load("pretrained_models/Cnn14.pth", map_location=self.device)

        if isinstance(checkpoint, dict) and "model" in checkpoint:
            state_dict = checkpoint["model"]
        else:
            state_dict = checkpoint
        
        model.load_state_dict(state_dict, strict=False)
        print(f"✓ Loaded pretrained CNN14 weights")

        # Replace final layer
        model.fc_audioset = nn.Linear(2048, self.esc50_num_classes)
        print(f"✓ Replaced final layer: {self.esc50_num_classes} classes")

        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total_params = sum(p.numel() for p in model.parameters())
        print(f"✓ All layers trainable initially")
        print(f"✓ Trainable: {trainable_params:,} / {total_params:,} "
              f"({100 * trainable_params / total_params:.1f}%)")
        print("="*70 + "\n")

        return model.to(self.device)

    def load_esc50_trained_model(self):

        print("\n" + "="*70)
        print("LOADING ESC-50 TRAINED MODEL FOR GENRES")
        print("="*70)

        # Create model with ESC-50 classes
        model = Cnn14(classes_num=50)
        
        # Load ESC-50 weights
        checkpoint = torch.load("best_esc50_model.pth", map_location=self.device)
        model.load_state_dict(checkpoint)
        print(f"✓ Loaded ESC-50 model (trained on 5-second clips)")
        
        # Replace for genres
        model.fc_audioset = nn.Linear(2048, self.num_genres)
        print(f"✓ Replaced final layer: 2048 -> {self.num_genres} genres")
        
        # Freeze all except classifier
        for name, param in model.named_parameters():
            if 'fc_audioset' not in name:
                param.requires_grad = False
        
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total_params = sum(p.numel() for p in model.parameters())
        print(f"✓ All layers trainable initially")
        print(f"✓ Trainable: {trainable_params:,} / {total_params:,}")
        print("="*70 + "\n")
        
        return model.to(self.device)
    

    def complete_training(self):
        
        model_esc50 = self.load_pretrained_cnn14()
        
        esc50_train_dataloader, esc50_test_dataloader = self.get_esc50_dataloader()

        _ = train_esc50_model(
            model=model_esc50,
            device=self.device,
            epochs=self.esc50_epochs,
            train_dataloader=esc50_train_dataloader,
            test_dataloader=esc50_test_dataloader
        )

        del model_esc50, esc50_train_dataloader, esc50_test_dataloader
        if self.device == "cuda":
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
            import gc
            gc.collect()

        model_genre = self.load_esc50_trained_model()

        genre_train_dataloader, genre_test_datalaoder = self.get_genre_dataloader()

        _ = train_genre_model(
            model=model_genre,
            device=self.device,
            epochs=self.genre_epochs,
            train_dataloader=genre_train_dataloader,
            test_dataloader=genre_test_datalaoder
        )


In [9]:
model_pipeline = ModelPipeline(
    esc50_dir=ESC_50_MASTER,
    genre_dir=GENRES_STEMS
)

model_pipeline.complete_training()


LOADING PRETRAINED CNN14 FOR ESC-50
✓ Loaded pretrained CNN14 weights
✓ Replaced final layer: 50 classes
✓ All layers trainable initially
✓ Trainable: 79,776,242 / 80,859,698 (98.7%)

Loading ESC-50 dataset...
Loading metadata from file: /kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/meta/esc50.csv
Loading metadata from file: /kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/meta/esc50.csv

ESC-50 TRAINING


3. Setting up training...
STAGE 1: Training only final layer (lr=0.001)
         Trainable params: 102,450



/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be r

Epoch [  1/30] Train Loss: 3.5275 | Train Acc:  33.50% | Val Loss: 2.9469 | Val Acc:  78.75%
         ✓ Saved best model (val_acc: 78.75%)
Epoch [  2/30] Train Loss: 2.6247 | Train Acc:  75.81% | Val Loss: 2.2041 | Val Acc:  83.50%
         ✓ Saved best model (val_acc: 83.50%)
Epoch [  3/30] Train Loss: 2.0074 | Train Acc:  81.25% | Val Loss: 1.7215 | Val Acc:  84.50%
         ✓ Saved best model (val_acc: 84.50%)
Epoch [  4/30] Train Loss: 1.5899 | Train Acc:  84.44% | Val Loss: 1.3840 | Val Acc:  87.00%
         ✓ Saved best model (val_acc: 87.00%)
Epoch [  5/30] Train Loss: 1.3519 | Train Acc:  83.50% | Val Loss: 1.1931 | Val Acc:  87.00%
         ✓ Saved best model (val_acc: 87.00%)
Epoch [  6/30] Train Loss: 1.1569 | Train Acc:  85.81% | Val Loss: 1.0370 | Val Acc:  86.50%
Epoch [  7/30] Train Loss: 1.0687 | Train Acc:  86.06% | Val Loss: 0.9695 | Val Acc:  86.25%
Epoch [  8/30] Train Loss: 0.8879 | Train Acc:  87.81% | Val Loss: 0.8371 | Val Acc:  88.50%
         ✓ Saved best mode